# App-1b : Le problème des N-Reines — Jumeau C#

> **Twin C#** de [`App-1-NQueens`](App-1-NQueens.ipynb) (Python + OR-Tools + numpy + matplotlib).
> Suite du marathon **.NET ⇄ Python** (#4956), volet **Search / Applications / CSP**.
> BCL .NET seule, **0 NuGet**, **from-scratch** (Prong B #3801).

## Objectifs d'apprentissage

1. Modéliser les N-Reines comme un **CSP** (variables, domaines, contraintes).
2. Comparer **trois familles** de résolution : *backtracking* (avec heuristiques MRV + Forward Checking), *min-conflicts* (recherche locale), et **énumération complète** (toutes les solutions + comptage des solutions fondamentales par brisure de symétrie).
3. Comprendre **algorithmiquement** le « miracle » de Min-Conflicts (passage à l'échelle jusqu'à N = 1000+) et la **brisure de symétrie** (8 transformations du carré → forme canonique).

## Complémentarité (#3801 Prong B)

| Twin | Outil | Valeur |
|------|-------|--------|
| Python | **OR-Tools CP-SAT** (solveur industriel) + numpy + matplotlib | invoquer un solveur, simulation vectorielle |
| **Ce twin (.NET BCL seule)** | **from-scratch** (récursion, propagation de domaines, marche aléatoire, transforms D4) | **comprendre la mécanique** : backtracking, propagation, recherche locale, symétries |

Le twin Python **appelle** un solveur ; ce twin **déroule** les algorithmes. Les deux se complètent : on voit ici *ce que fait* CP-SAT sous le capot (propagation, énumération, brisure de symétrie).

**Durée estimée : 35 min.** Prérequis : [`Search-1`](../../Part1-Foundations/Search-1-UninformedSearch.ipynb), [`Search-9`](../../Part2-CSP/Search-9-ConstraintSatisfaction.ipynb) (CSP).


## 1. Représentation du problème

On représente un plateau par `queens[col] = row` : **une reine par colonne** (la contrainte « une par colonne » est donc satisfaite par construction). Il reste à vérifier les conflits de **ligne** et de **diagonales**.

Cellule d'installation : on définit le helper de formatage invariant (pour éviter la collision virgule-décimale FR) et les compteurs de nœuds explorés.


In [1]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;

// Formatage invariant (règle marathon : la culture FR ne persiste pas entre cellules .NET Interactive)
static string FI(double x, string fmt = "F3") => x.ToString(fmt, CultureInfo.InvariantCulture);
static string FI(long x) => x.ToString("N0", CultureInfo.InvariantCulture);
static void Show(string s) => display(s);

// conflit entre (col,row) et les reines déjà placées dans les colonnes marquées `placed`
static bool IsSafe(int[] queens, int col, int row, bool[] placed) {
    for (int c = 0; c < queens.Length; c++) {
        if (c == col || !placed[c]) continue;
        int r = queens[c];
        if (r == row) return false;                       // même ligne
        if (Math.Abs(r - row) == Math.Abs(col - c)) return false;  // même diagonale
    }
    return true;
}

Show("Setup OK — représentation queens[col]=row, helper IsSafe défini.");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Setup OK — représentation queens[col]=row, helper IsSafe défini.

## 2. Approche 1a : Backtracking simple

On remplit les colonnes de gauche à droite. Pour chaque colonne, on essaie les lignes de haut en bas ; la première ligne **sûre** est acceptée et on descend. Si aucune ligne ne convient, on **remonte** (backtrack).

C'est le DFS sur l'arbre des affectations partielles. On compte les **nœuds explorés** pour comparer les variantes.

In [2]:
long _btNodes;

bool BacktrackSimple(int[] queens, bool[] placed, int col, int n) {
    if (col == n) return true;           // toutes les colonnes placées -> solution
    _btNodes++;
    for (int row = 0; row < n; row++) {
        if (IsSafe(queens, col, row, placed)) {
            queens[col] = row; placed[col] = true;
            if (BacktrackSimple(queens, placed, col + 1, n)) return true;
            placed[col] = false;          // backtrack
        }
    }
    return false;
}

int SolveSimple(int n) {
    var queens = new int[n]; var placed = new bool[n]; _btNodes = 0;
    BacktrackSimple(queens, placed, 0, n);
    return _btNodes > 0 ? 1 : 0;
}

string BoardString(int[] queens) {
    int n = queens.Length; var sb = new System.Text.StringBuilder();
    for (int r = 0; r < n; r++) {
        for (int c = 0; c < n; c++) sb.Append(queens[c] == r ? "Q " : ". ");
        sb.AppendLine();
    }
    return sb.ToString();
}

int n = 8;
var q = new int[n]; var p = new bool[n]; _btNodes = 0;
BacktrackSimple(q, p, 0, n);
Show("Backtracking simple, N = " + n + " -> " + FI(_btNodes) + " nœuds explorés.");
Show(BoardString(q));


Backtracking simple, N = 8 -> 113 nœuds explorés.

Q . . . . . . . 
. . . . . . Q . 
. . . . Q . . . 
. . . . . . . Q 
. Q . . . . . . 
. . . Q . . . . 
. . . . . Q . . 
. . Q . . . . . 


### 2.1 Backtracking avec heuristique MRV (Minimum Remaining Values)

Au lieu de fixer l'ordre des colonnes, on choisit à chaque pas la colonne **non assignée** qui a le **moins** de lignes sûres possibles. Intuition : échouer vite là où on est le plus contraint réduit la taille de l'arbre.

> Ici, comme chaque colonne reçoit exactement une reine, MRV agit sur l'**ordre des variables** (quelle colonne remplir ensuite), et non sur les domaines.

In [3]:
long _mrvNodes;

bool BacktrackMRV(int[] queens, bool[] placed, int n) {
    // 1. choisir la colonne non assignée avec le moins de lignes sûres (MRV)
    int bestCol = -1, bestCount = int.MaxValue;
    List<int> bestRows = new List<int>();
    for (int col = 0; col < n; col++) {
        if (placed[col]) continue;
        var rows = new List<int>();
        for (int row = 0; row < n; row++)
            if (IsSafe(queens, col, row, placed)) rows.Add(row);
        if (rows.Count < bestCount) { bestCount = rows.Count; bestCol = col; bestRows = rows; }
        if (bestCount == 0) return false;            // impasse détectée tôt
    }
    if (bestCol == -1) return true;                  // tout est assigné -> solution
    _mrvNodes++;
    placed[bestCol] = true;
    foreach (var row in bestRows) {
        queens[bestCol] = row;
        if (BacktrackMRV(queens, placed, n)) return true;
    }
    placed[bestCol] = false;
    return false;
}

int n = 8;
var q = new int[n]; var p = new bool[n]; _mrvNodes = 0;
BacktrackMRV(q, p, n);
Show("Backtracking MRV, N = " + n + " -> " + FI(_mrvNodes) + " nœuds explorés (vs simple ci-dessus).");


Backtracking MRV, N = 8 -> 52 nœuds explorés (vs simple ci-dessus).

### 2.2 Backtracking avec Forward Checking (FC)

À chaque placement, on **propage** : on retire des domaines des colonnes encore vides toutes les lignes devenues conflictuelles. Si un domaine devient vide, on déclenche le backtrack **immédiatement** (avant même de développer). On restaure les domaines au backtrack (pile des retraits).

C'est exactement la propagation de contraintes qu'effectue CP-SAT sous le capot.

In [4]:
long _fcNodes;

// domaines : un HashSet par colonne des lignes encore possibles
bool BacktrackFC(int[] queens, HashSet<int>[] domains, int placedCount, int n) {
    if (placedCount == n) return true;
    _fcNodes++;
    // choisir la colonne non assignée au plus petit domaine (MRV sur domaines filtrés)
    int bestCol = -1, bestSize = int.MaxValue;
    for (int col = 0; col < n; col++) {
        if (queens[col] >= 0) continue;
        if (domains[col].Count < bestSize) { bestSize = domains[col].Count; bestCol = col; }
    }
    if (bestSize == 0) return false;                 // domaine vide -> impasse
    // snapshot des lignes candidats (on itère sur une copie car on va modifier domains)
    var candidates = domains[bestCol].ToList();
    foreach (var row in candidates) {
        // tenter (bestCol, row) : propager
        var removed = new List<(int col, int row)>();
        bool dead = false;
        queens[bestCol] = row;
        for (int col = 0; col < n; col++) {
            if (col == bestCol || queens[col] >= 0) continue;
            var toRemove = domains[col].Where(r => r == row || Math.Abs(r - row) == Math.Abs(col - bestCol)).ToList();
            foreach (var r in toRemove) { domains[col].Remove(r); removed.Add((col, r)); }
            if (domains[col].Count == 0) { dead = true; break; }
        }
        if (!dead && BacktrackFC(queens, domains, placedCount + 1, n)) return true;
        // restaurer
        foreach (var (c, r) in removed) domains[c].Add(r);
        queens[bestCol] = -1;
    }
    return false;
}

int n = 8;
var q = Enumerable.Range(0, n).Select(_ => -1).ToArray();
var dom = Enumerable.Range(0, n).Select(_ => new HashSet<int>(Enumerable.Range(0, n))).ToArray();
_fcNodes = 0;
BacktrackFC(q, dom, 0, n);
Show("Backtracking Forward-Checking, N = " + n + " -> " + FI(_fcNodes) + " nœuds explorés.");
Show(BoardString(q));


Backtracking Forward-Checking, N = 8 -> 51 nœuds explorés.

Q . . . . . . . 
. . . . . . Q . 
. . . . Q . . . 
. . . . . . . Q 
. Q . . . . . . 
. . . Q . . . . 
. . . . . Q . . 
. . Q . . . . . 


### 2.3 Benchmark des trois variantes

On compare le **nombre de nœuds explorés** et le **temps** pour trouver une première solution, de N = 8 à N = 16. Attendu : FC < MRV < simple (la propagation coupe l'arbre plus tôt).

In [5]:
var results = new List<(int n, long simple, long mrv, long fc, double tSimple, double tMrv, double tFc)>();
for (int n = 8; n <= 16; n += 2) {
    // simple
    var q1 = new int[n]; var p1 = new bool[n]; _btNodes = 0;
    var sw = Stopwatch.StartNew(); BacktrackSimple(q1, p1, 0, n); sw.Stop();
    double ts = sw.Elapsed.TotalMilliseconds; long ns = _btNodes;
    // MRV
    var q2 = new int[n]; var p2 = new bool[n]; _mrvNodes = 0;
    sw = Stopwatch.StartNew(); BacktrackMRV(q2, p2, n); sw.Stop();
    double tm = sw.Elapsed.TotalMilliseconds; long nm = _mrvNodes;
    // FC
    var q3 = Enumerable.Range(0, n).Select(_ => -1).ToArray();
    var dom3 = Enumerable.Range(0, n).Select(_ => new HashSet<int>(Enumerable.Range(0, n))).ToArray();
    _fcNodes = 0;
    sw = Stopwatch.StartNew(); BacktrackFC(q3, dom3, 0, n); sw.Stop();
    double tf = sw.Elapsed.TotalMilliseconds; long nf = _fcNodes;
    results.Add((n, ns, nm, nf, ts, tm, tf));
}

var hdr = "N".PadLeft(4) + " | " + "simple".PadLeft(10) + " | " + "MRV".PadLeft(10) + " | " + "FC".PadLeft(10) + " | " + "t_simple(ms)".PadLeft(12) + " | " + "t_MRV(ms)".PadLeft(10) + " | " + "t_FC(ms)".PadLeft(9);
Show(hdr);
Show(new string('-', hdr.Length));
foreach (var r in results) {
    string line = r.n.ToString().PadLeft(4) + " | " + FI(r.simple).PadLeft(10) + " | " + FI(r.mrv).PadLeft(10) + " | "
                + FI(r.fc).PadLeft(10) + " | " + FI(r.tSimple, "F3").PadLeft(12) + " | " + FI(r.tMrv, "F3").PadLeft(10) + " | " + FI(r.tFc, "F3").PadLeft(9);
    Show(line);
}
Show("Interprétation : simple explose (croissance ~factorielle), MRV et FC la divisent par 1-2 ordres de grandeur.");
Show("MRV et FC sont comparables : selon l'instance, l'un ou l'autre gagne (la propagation de FC et");
Show("l'ordonnancement de MRV coupent l'arbre différemment). Le gain dominant vient de l'heuristique vs simple.");


   N |     simple |        MRV |         FC | t_simple(ms) |  t_MRV(ms) |  t_FC(ms)

-----------------------------------------------------------------------------------

   8 |        113 |         52 |         51 |        0.029 |      0.089 |     0.109

  10 |        102 |         26 |         26 |        0.032 |      0.067 |     0.051

  12 |        261 |        107 |         84 |        0.109 |      0.347 |     0.165

  14 |      1,899 |         91 |        105 |        0.885 |      0.466 |     0.235

  16 |     10,052 |         37 |         37 |        6.113 |      0.246 |     0.097

Interprétation : simple explose (croissance ~factorielle), MRV et FC la divisent par 1-2 ordres de grandeur.

MRV et FC sont comparables : selon l'instance, l'un ou l'autre gagne (la propagation de FC et

l'ordonnancement de MRV coupent l'arbre différemment). Le gain dominant vient de l'heuristique vs simple.

## 3. Approche 2 : Min-Conflicts (recherche locale)

idée radicalement différente : on part d'une affectation **aléatoire** (une reine par colonne, ligne aléatoire), puis on **répare**. À chaque itération, on choisit une colonne en conflit et on déplace sa reine vers la ligne qui **minimise** le nombre de conflits. Pas de retour arrière : c'est une **marche** dans l'espace des configurations.

C'est l'ancêtre des métaheuristiques (proche du hill-climbing stochastique, cf [`Search-11`](../../Part3-Advanced/Search-11-Metaheuristics.ipynb)).

In [6]:
// Conflits maintenus incrémentalement via compteurs par ligne / diagonale (O(1) par requête)
//  - rowConf[r]      = nombre de reines dans la ligne r
//  - diag1Conf[r-c+n]= nombre de reines sur la diagonale NO-SE (différence r-c constante)
//  - diag2Conf[r+c]  = nombre de reines sur la diagonale SO-NE (somme r+c constante)
// Conflits subis par la reine (col,row) = (rowConf[row]-1) + (diag1-1) + (diag2-1)
(bool ok, int iters, int[] queens) MinConflicts(int n, int maxIters, int seed) {
    var rnd = new Random(seed);
    var queens = new int[n];
    var rowConf = new int[n];
    var diag1 = new int[2 * n + 1];   // indice r-c+n in [1..2n-1]
    var diag2 = new int[2 * n + 1];   // indice r+c in [0..2n-2]
    for (int c = 0; c < n; c++) {
        int r = rnd.Next(n);
        queens[c] = r;
        rowConf[r]++; diag1[r - c + n]++; diag2[r + c]++;
    }
    for (int it = 0; it < maxIters; it++) {
        // colonnes en conflit
        var conflicted = new List<int>();
        for (int c = 0; c < n; c++) {
            int r = queens[c];
            int conf = (rowConf[r] - 1) + (diag1[r - c + n] - 1) + (diag2[r + c] - 1);
            if (conf > 0) conflicted.Add(c);
        }
        if (conflicted.Count == 0) return (true, it, queens);
        int col = conflicted[rnd.Next(conflicted.Count)];
        int curRow = queens[col];
        // collecter TOUTES les lignes minimisant les conflits, puis tirage uniforme
        // (tie-break uniforme = évite les cycles ; la reine déplacée EXCLUE de son propre compte)
        int bestConf = int.MaxValue;
        var bestRows = new List<int>();
        for (int row = 0; row < n; row++) {
            int conf = rowConf[row] + diag1[row - col + n] + diag2[row + col];
            if (row == curRow) conf -= 3;            // elle occupe ces 3 compteurs
            if (conf < bestConf) { bestConf = conf; bestRows.Clear(); bestRows.Add(row); }
            else if (conf == bestConf) bestRows.Add(row);
        }
        int bestRow = bestRows[rnd.Next(bestRows.Count)];
        if (bestRow != curRow) {
            rowConf[curRow]--; diag1[curRow - col + n]--; diag2[curRow + col]--;
            queens[col] = bestRow;
            rowConf[bestRow]++; diag1[bestRow - col + n]++; diag2[bestRow + col]++;
        }
    }
    int cc = 0;
    for (int c = 0; c < n; c++) { int r = queens[c]; cc += (rowConf[r]-1) + (diag1[r-c+n]-1) + (diag2[r+c]-1); }
    return (cc == 0, maxIters, queens);
}

int n = 8;
var (ok, iters, sol) = MinConflicts(n, 10000, 42);
Show("Min-Conflicts, N = " + n + " -> solution en " + iters + " itérations (ok=" + ok + ").");
if (ok) Show(BoardString(sol));


Min-Conflicts, N = 8 -> solution en 17 itérations (ok=True).

. . Q . . . . . 
. . . . . Q . . 
. . . . . . . Q 
. Q . . . . . . 
. . . Q . . . . 
Q . . . . . . . 
. . . . . . Q . 
. . . . Q . . . 


### 3.1 Le « miracle » de Min-Conflicts : passage à l'échelle

Le backtracking systématique explose (factoriel). Min-Conflicts, lui, résout des instances **énormes** en quelques centaines d'itérations. C'est le résultat contre-intuitif de la recherche locale : sur les N-Reines (problème peu contraint à grande échelle), elle **surclasse** la recherche complète.

In [7]:
var sizes = new[] { 8, 50, 100, 200, 500, 1000 };
Show("N".PadLeft(6) + " | " + "résolu".PadLeft(6) + " | " + "itérations".PadLeft(11) + " | " + "temps(ms)".PadLeft(10));
Show(new string('-', 42));
foreach (var n in sizes) {
    var sw = Stopwatch.StartNew();
    var (ok, it, _) = MinConflicts(n, 20000, 7);
    sw.Stop();
    Show(n.ToString().PadLeft(6) + " | " + (ok ? "oui" : "non").PadLeft(6) + " | " + (ok ? it.ToString() : ">max").PadLeft(11) + " | " + FI(sw.Elapsed.TotalMilliseconds, "F1").PadLeft(10));
}
Show("Interprétation : N = 1000 résolu en quelques centaines d'itérations — inaccessible au backtracking.");


     N | résolu |  itérations |  temps(ms)

------------------------------------------

     8 |    oui |          34 |        0.0

    50 |    oui |         117 |        0.1

   100 |    oui |         166 |        0.2

   200 |    oui |         212 |        0.4

   500 |    oui |         388 |        3.1

  1000 |    oui |         735 |        6.3

Interprétation : N = 1000 résolu en quelques centaines d'itérations — inaccessible au backtracking.

## 4. Approche 3 : Énumération de toutes les solutions

Min-Conflicts donne **une** solution. Pour **compter** les solutions (N = 8 → 92), il faut un parcours complet : on retire le `return true` du backtracking et on incrémente un compteur à chaque feuille.

In [8]:
long _solCount;
void EnumerateAll(int[] queens, bool[] placed, int col, int n) {
    if (col == n) { _solCount++; return; }
    for (int row = 0; row < n; row++) {
        if (IsSafe(queens, col, row, placed)) {
            queens[col] = row; placed[col] = true;
            EnumerateAll(queens, placed, col + 1, n);
            placed[col] = false;
        }
    }
}

int n = 8;
var q = new int[n]; var p = new bool[n]; _solCount = 0;
EnumerateAll(q, p, 0, n);
Show("N = " + n + " -> " + _solCount + " solutions distinctes (résultat attendu : 92).");
// petites valeurs de référence
Show("Référence OEIS A000170 : N=1..8 = 1, 0, 0, 2, 10, 4, 40, 92.");


N = 8 -> 92 solutions distinctes (résultat attendu : 92).

Référence OEIS A000170 : N=1..8 = 1, 0, 0, 2, 10, 4, 40, 92.

### 4.1 Brisure de symétrie : solutions fondamentales

Beaucoup des 92 solutions sont **équivalentes** par une symétrie du carré (le groupe diédral D4 : 4 rotations × 2 réflexions = 8 transformations). Deux solutions « fondamentalement identiques » donnent la même **forme canonique** = la plus petite (lexicalement) parmi leurs 8 transformées.

On obtient ainsi les **12 solutions fondamentales** de N = 8. C'est exactement la brisure de symétrie qu'OR-Tools CP-SAT peut ajouter déclarativement (contraintes `AddAllDifferent` + lex-ordering) — ici on la déroule à la main.

In [9]:
int[] Transform(int[] q, string kind) {
    int n = q.Length;
    var nq = new int[n];
    for (int x = 0; x < n; x++) {
        int y = q[x]; int nx, ny;
        switch (kind) {
            case "id":   nx = x; ny = y; break;
            case "r90":  nx = n - 1 - y; ny = x; break;
            case "r180": nx = n - 1 - x; ny = n - 1 - y; break;
            case "r270": nx = y; ny = n - 1 - x; break;
            case "fv":   nx = n - 1 - x; ny = y; break;   // réflexion verticale
            case "fh":   nx = x; ny = n - 1 - y; break;    // réflexion horizontale
            case "fd":   nx = y; ny = x; break;            // diagonale principale
            default:     nx = n - 1 - y; ny = n - 1 - x; break; // anti-diagonale
        }
        nq[nx] = ny;
    }
    return nq;
}
string Key(int[] q) => string.Join(",", q);
string Canonical(int[] q) {
    var forms = new[] { "id", "r90", "r180", "r270", "fv", "fh", "fd", "fa" }
        .Select(k => Transform(q, k)).Select(Key).OrderBy(s => s);
    return forms.First();
}

// collecter toutes les solutions puis dénombrer les formes canoniques distinctes
int n = 8;
var all = new List<int[]>();
void Collect(int[] queens, bool[] placed, int col) {
    if (col == n) { all.Add((int[])queens.Clone()); return; }
    for (int row = 0; row < n; row++) {
        if (IsSafe(queens, col, row, placed)) {
            queens[col] = row; placed[col] = true;
            Collect(queens, placed, col + 1);
            placed[col] = false;
        }
    }
}
var q = new int[n]; var p = new bool[n];
Collect(q, p, 0);

var canonicals = new HashSet<string>();
foreach (var sol in all) canonicals.Add(Canonical(sol));
Show("N = " + n + " : " + all.Count + " solutions, " + canonicals.Count + " fondamentales (attendu : 12).");
Show("Les " + (all.Count - canonicals.Count) + " autres s'obtiennent par symétrie du carré (groupe D4, ordre 8).");


N = 8 : 92 solutions, 12 fondamentales (attendu : 12).

Les 80 autres s'obtiennent par symétrie du carré (groupe D4, ordre 8).

## Synthèse comparée

| Approche | Type | Garantit solution ? | Compte toutes les solutions ? | Passe à l'échelle ? |
|----------|------|---------------------|-------------------------------|---------------------|
| Backtracking simple | recherche complète | oui | non (1 solution) | non (factoriel) |
| + MRV | complète + heuristique | oui | non | un peu mieux |
| + Forward Checking | complète + propagation | oui | non | nettement mieux |
| Min-Conflicts | recherche locale | **non** (stochastique) | non | **oui (N = 1000+)** |
| Énumération complète | parcours total | oui | **oui** | non (explosif) |
| + brisure de symétrie | parcours + D4 | oui | **fondamentales** | non |

**Leçon** : aucun algorithme ne domine partout. La recherche complète garantit mais explose ; la recherche locale passe à l'échelle mais ne prouve rien. Les solveurs industriels (CP-SAT) combinent propagation + recherche + brisure de symétrie déclarative.

In [10]:
Show("Synthèse affichée ci-dessus (tableau markdown).");

Synthèse affichée ci-dessus (tableau markdown).

## 6. Exercices

### Exercice 1 : le problème des N-Tours

Les tours ne conflituent que sur la **ligne et la colonne** (pas de diagonales). Avec une tour par colonne (contrainte déjà satisfaite), il ne reste qu'à placer une tour par **ligne**. Compter les solutions pour N = 8 — c'est simplement le nombre de **permutations** de N lignes.

> **Indice** : `IsSafe` devient `r == row` seulement (retirer la diagonale). Le compte attendu est `N!`.

In [11]:
// Exercice 1 : N-Tours — une tour par colonne ET par ligne (pas de diagonale)
// Etape 1 : adapter IsSafe (retirer la condition de diagonale).
// Etape 2 : énumérer les solutions via EnumerateAll modifié.
// Etape 3 : vérifier que le compte vaut N! pour N = 8 (40320).
static long Factorial(int k) { long f = 1; for (int i = 2; i <= k; i++) f *= i; return f; }
Show("Exercice 1 à compléter — attendu N=8 : " + Factorial(8) + " (8!).");


Exercice 1 à compléter — attendu N=8 : 40320 (8!).

### Exercice 2 : seuil de bascule backtracking / min-conflicts

Pour petit N, le backtracking est instantané ; pour grand N, seul Min-Conflicts passe. Où se situe le **seuil** au-delà duquel Min-Conflicts devient plus rapide (en temps) que le backtracking avec Forward Checking ?

> **Indice** : boucler sur N de 8 à 30, mesurer `t_FC` et `t_MC`, trouver le plus petit N tel que `t_MC < t_FC`.

In [12]:
// Exercice 2 : trouver le plus petit N tel que t_MinConflicts < t_ForwardChecking
// Etape 1 : pour N = 8..30, mesurer les deux temps (réutiliser les fonctions ci-dessus).
// Etape 2 : retourner le premier N où MC bat FC.
Show("Exercice 2 à compléter — mesurer le crossover FC/MC.");


Exercice 2 à compléter — mesurer le crossover FC/MC.

### Exercice 3 : briser la symétrie directement dans l'énumération

Plutôt que d'énumérer les 92 solutions puis de filtrer par forme canonique (ce qui coûte 92 × 8 transforms), on peut **imposer** dès la descente que la solution soit la forme canonique (ex. contrainte lexicographique `queens < Transform(queens, "r90")`). Implémenter cette brisure **à la volée**.

> **Indice** : à la feuille (col == n), ne compter la solution que si son `Key` est égal à son `Canonical`. C'est une brisure *a posteriori* par feuille ; la brisure *a priori* (couper pendant la descente) est plus dure — c'est précisément ce que fait CP-SAT.

In [13]:
// Exercice 3 : énumérer en ne gardant que les formes canoniques (a posteriori par feuille)
// Etape 1 : modifier EnumerateAll pour, à la feuille, vérifier Key(queens) == Canonical(queens).
// Etape 2 : le compte doit donner directement 12 pour N = 8 (sans HashSet).
Show("Exercice 3 à compléter — briser la symétrie à l'énumération.");


Exercice 3 à compléter — briser la symétrie à l'énumération.

## Conclusion

Ce twin C# a **déroulé à la main** ce que le solveur industriel OR-Tools CP-SAT (twin Python) automatise :

- **Backtracking** (simple / MRV / Forward Checking) = le moteur de recherche complète avec propagation ;
- **Min-Conflicts** = la recherche locale qui passe à l'échelle (le « miracle » N = 1000) ;
- **Énumération + brisure de symétrie D4** = le comptage exact des solutions fondamentales (12 pour N = 8).

**Lien avec les Foundations** : backtracking ⇄ [`Search-1`](../../Part1-Foundations/Search-1-UninformedSearch.ipynb) (DFS), CSP ⇄ [`Search-9`](../../Part2-CSP/Search-9-ConstraintSatisfaction.ipynb), Min-Conflicts ⇄ [`Search-11`](../../Part3-Advanced/Search-11-Metaheuristics.ipynb) (métaheuristiques), CP-SAT ⇄ [`App-1-NQueens`](App-1-NQueens.ipynb) (twin Python, solveur industriel).

### Pour aller plus loin
- Ajouter la **propagation AC-3** (arc-consistency) avant le backtracking.
- Comparer au **recuit simulé** (cf Search-11 SA) sur les mêmes instances.
- Mesurer l'impact de la **brisure de symétrie déclarative** sur le temps d'énumération.

---
*Marathon .NET ⇄ Python #4956 — Search/Applications, tranche N-Reines. See #4956, #3801.*
